<a href="https://colab.research.google.com/github/ZahiaYanes/medmcqa-lora-expert-fusion/blob/main/01_baseline_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!pip install datasets transformers peft accelerate bitsandbytes -q

## Data Loading

We use the MedMCQA dataset (Pal et al.), which contains ~194k multiple-choice
medical questions organized by subject. Unlike MedQA (USMLE-style), MedMCQA
natively provides a `subject_name` field, which allows us to split questions
by domain — a key requirement for training domain-specific "expert" models.

In [2]:
from datasets import load_dataset

dataset = load_dataset("openlifescienceai/medmcqa")
print(dataset)

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.9MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  936kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4183 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 182822
    })
    test: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 6150
    })
    validation: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 4183
    })
})


In [3]:
print(dataset["train"][0])

{'id': 'e9ad821a-c438-4965-9f77-760819dfa155', 'question': 'Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma', 'opa': 'Hyperplasia', 'opb': 'Hyperophy', 'opc': 'Atrophy', 'opd': 'Dyplasia', 'cop': 2, 'choice_type': 'single', 'exp': 'Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calculus associated with progressive atrophy of the kidney due to obstruction to the outflow of urine Refer Robbins 7yh/9,1012,9/e. P950', 'subject_name': 'Anatomy', 'topic_name': 'Urinary tract'}


In [4]:
import pandas as pd

df_train = dataset["train"].to_pandas()
print(df_train["subject_name"].value_counts())

subject_name
Medicine                        17887
Surgery                         16862
Pathology                       14884
Anatomy                         14560
Pharmacology                    13758
Social & Preventive Medicine    11882
Microbiology                    11314
Gynaecology & Obstetrics        10013
Dental                           8938
Physiology                       8830
Biochemistry                     8282
Pediatrics                       8037
Ophthalmology                    6932
Forensic Medicine                5900
ENT                              4919
Psychiatry                       4442
Radiology                        4395
Anaesthesia                      3172
Unknown                          3045
Orthopaedics                     2999
Skin                             1771
Name: count, dtype: int64


## Note on data splits

The `test` split has all `cop` (correct option) values set to `-1`. This is
intentional: it prevents users from training or evaluating directly against
the test labels, which are typically reserved for an external leaderboard.

For local evaluation during development, we instead use the `validation`
split, which contains real, usable labels.

In [5]:
df_test = dataset["test"].to_pandas()
print(df_test["cop"].value_counts())
print(df_test[df_test["subject_name"] == "Pharmacology"].head(3))

cop
-1    6150
Name: count, dtype: int64
                                      id  \
2   f6ce5597-c646-4a2b-8767-764f185be603   
13  84bdd06e-9593-4f59-bf53-0b68652bcb89   
34  85a4856a-40bb-44a1-a025-3278b8faa922   

                                             question           opa  \
2   Which macrolide is active against Mycobaterium...  Azithromycin   
13   FDA approved drug for refractory schizophrenia ?     Amoxapine   
34                           Orange sweat is seen in:      Rifampin   

              opb             opc           opd  cop choice_type exp  \
2   Roxithromycin  Clarithromycin    Framycetin   -1      single       
13    Haloperidol       Clozapine   Penfluridol   -1      single       
34            INH   Thioacetazone  Pyrazinamide   -1      single       

    subject_name topic_name  
2   Pharmacology       None  
13  Pharmacology       None  
34  Pharmacology       None  


In [6]:
df_val = dataset["validation"].to_pandas()
print(df_val["cop"].value_counts())
print(df_val["subject_name"].value_counts())

cop
0    1348
1    1085
2     925
3     825
Name: count, dtype: int64
subject_name
Dental                          1318
Surgery                          369
Pathology                        337
Medicine                         295
Pharmacology                     243
Pediatrics                       234
Anatomy                          234
Gynaecology & Obstetrics         224
Physiology                       171
Biochemistry                     171
Social & Preventive Medicine     129
Microbiology                     122
Radiology                         69
Forensic Medicine                 67
Ophthalmology                     58
ENT                               53
Anaesthesia                       34
Orthopaedics                      20
Skin                              17
Psychiatry                        16
Unknown                            2
Name: count, dtype: int64


In [7]:
 # Load model (Qwen2.5-1.5B-Instruct)
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # use half precision to reduce memory usage
    device_map="auto"           # automatically place the model on the available GPU
)

print("Model loaded successfully!")
print("Device used:", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Device used: cuda:0


In [8]:
def format_prompt(question, opa, opb, opc, opd):
    # Build a simple instruction prompt for a 4-choice MCQ
    prompt = f"""Answer the following medical question by responding with only the letter of the correct option (A, B, C, or D).

Question: {question}
A) {opa}
B) {opb}
C) {opc}
D) {opd}

Answer:"""
    return prompt

def get_model_answer(question, opa, opb, opc, opd):
    prompt = format_prompt(question, opa, opb, opc, opd)
    messages = [{"role": "user", "content": prompt}]

    # Apply the chat template expected by the Instruct model
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5,   # we only need a single letter
        do_sample=False     # deterministic output, for reproducible evaluation
    )

    # Decode only the newly generated tokens (skip the input prompt)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

In [9]:
# Single example test
example = dataset["validation"][0]
print("Question:", example["question"])
print("Expected answer (index):", example["cop"])

answer = get_model_answer(example["question"], example["opa"], example["opb"], example["opc"], example["opd"])
print("Model answer:", answer)

Question: Which of the following is not true for myelinated nerve fibers:
Expected answer (index): 0
Model answer: A)


In [10]:
def extract_letter(model_output):
    # Clean the model's raw output to extract a single letter (A, B, C, or D)
    model_output = model_output.strip().upper()
    for letter in ["A", "B", "C", "D"]:
        if model_output.startswith(letter):
            return letter
    return None  # could not parse a valid answer

def cop_to_letter(cop_index):
    # Convert the numeric correct-option index into a letter
    mapping = {0: "A", 1: "B", 2: "C", 3: "D"}
    return mapping.get(cop_index, None)

# Quick test on our previous example
predicted = extract_letter(answer)
expected = cop_to_letter(example["cop"])
print("Predicted:", predicted, "| Expected:", expected, "| Correct:", predicted == expected)

Predicted: A | Expected: A | Correct: True


## Baseline Evaluation

Before any fine-tuning, we evaluate the raw Qwen2.5-1.5B-Instruct model on our
three target domains (Pharmacology, Pathology, Surgery), using a random sample
of 50 questions per domain from the validation set.

This baseline will serve as our reference point: if LoRA fine-tuning is
effective, we expect accuracy to improve above these numbers.

In [11]:
import time

def evaluate_on_subject(dataset_split, subject_name, n_samples=50):
    # Filter the dataset to keep only questions from the target subject
    subject_data = [ex for ex in dataset_split if ex["subject_name"] == subject_name]
    subject_data = subject_data[:n_samples]  # limit sample size for speed

    correct = 0
    total = 0

    for example in subject_data:
        answer = get_model_answer(
            example["question"], example["opa"], example["opb"],
            example["opc"], example["opd"]
        )
        predicted = extract_letter(answer)
        expected = cop_to_letter(example["cop"])

        if predicted is not None:
            total += 1
            if predicted == expected:
                correct += 1

    accuracy = correct / total if total > 0 else 0
    return accuracy, correct, total

# Run baseline evaluation on our 3 target domains
subjects = ["Pharmacology", "Pathology", "Surgery"]
results = {}

for subject in subjects:
    start = time.time()
    accuracy, correct, total = evaluate_on_subject(dataset["validation"], subject, n_samples=50)
    elapsed = time.time() - start
    results[subject] = accuracy
    print(f"{subject}: {accuracy:.2%} ({correct}/{total} correct) — {elapsed:.1f}s")

Pharmacology: 52.00% (26/50 correct) — 11.3s
Pathology: 46.00% (23/50 correct) — 10.3s
Surgery: 38.00% (19/50 correct) — 11.1s


## Baseline Results

| Domain       | Accuracy | Correct/Total |
|--------------|----------|----------------|
| Pharmacology | 52.00%   | 26/50          |
| Pathology    | 46.00%   | 23/50          |
| Surgery      | 38.00%   | 19/50          |

For reference, random guessing on a 4-option MCQ would give ~25% accuracy.
The base model already performs above chance on all three domains, with
Pharmacology being the strongest and Surgery the weakest — likely reflecting
differences in how well-represented each domain is in the model's pretraining
data.

**Caveat**: with only 50 samples per domain, these estimates have a fairly
wide margin of error (~±7 percentage points at a 95% confidence level).
We will re-evaluate on a larger sample size for the final comparison.